<p align="center">
<a href="https://colab.research.google.com/github/md-ryhan-uddin/ai-lab-experiments/blob/main/experiment_06.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

<a href="https://kaggle.com/kernels/welcome?src=https://github.com/md-ryhan-uddin/ai-lab-experiments/blob/main/experiment_06.ipynb">
    <img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open In Kaggle" height="20"/>
</a>
</p>

## **Experiment 06:** Implementation and Performance Evaluation of Perceptron and Multilayer Perceptron (MLP) for Classification

**Course:** ICE 4111: Artificial Intelligence Lab  
**Program:** Bachelor of Science in Information and Communication Engineering (BSc in ICE)  
**Instructor:** Md. Ryhan Uddin

**Student Name:** ______________________________  
**Student ID:** ______________________________  
**Section/Batch:** ______________________________  
**Date of Submission:** ______________________________  


### **Quick Overview**

In this lab you will build a single-layer **Perceptron** and a **Multilayer Perceptron (MLP)** in TensorFlow/Keras, train both on the same standardized dataset with a held-out validation set and early stopping, and compare them using learning curves, confusion matrices, and ROC curves. Several plots in this notebook are built with **Plotly** and **ipywidgets**, so you can hover over points for exact values and interactively move a classification-threshold slider to see how it changes the MLP's predictions in real time.

### **Learning Objectives**

After completing this experiment, students will be able to:

- Build and train a single-layer perceptron for binary classification in TensorFlow/Keras.
- Build and train an MLP with hidden layers and non-linear activations.
- Use a validation split and early stopping to avoid overfitting.
- Compare learning curves, confusion matrices, and ROC-AUC between the two models.
- Interactively explore how the classification threshold affects precision, recall, and the confusion matrix.


## **Theory**

### Introduction

The perceptron is one of the earliest neural network models. It computes a weighted sum of inputs and applies a threshold or sigmoid-like activation to make a binary decision. The multilayer perceptron (MLP) extends this idea by adding one or more hidden layers so that the network can model non-linear relationships. These models are simple but important because they introduce the core neural network ideas of weights, activations, and gradient-based learning that underpin deeper architectures.

### Mathematical Background

For input vector $x$, weights $w$, and bias $b$, the perceptron computes

$$z = w^T x + b$$

A binary output is produced with a sigmoid activation

$$\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$$

During training, weights are updated using gradient descent on a loss function such as binary cross-entropy:

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n} \left[ y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i) \right]$$

In an MLP, the hidden layer output is $h = \phi(W_1 x + b_1)$, where $\phi$ is a non-linear activation such as ReLU, and the final prediction is $\hat{y} = \sigma(W_2 h + b_2)$ for binary classification. Gradients are propagated back through both layers using backpropagation.

### Important Concepts

A perceptron learns by repeatedly presenting training examples, computing the prediction error, and adjusting weights so that future predictions improve. Because it has no hidden layer, it can only represent a linear decision boundary. An MLP uses backpropagation to compute gradients through multiple layers and then updates parameters with an optimizer such as Adam. Because the hidden layers introduce non-linear transformations, MLPs can solve problems that are not linearly separable.

**Early stopping** is a practical regularization method that halts training when validation loss stops improving for a set number of epochs (the *patience*), and restores the weights from the best epoch. This prevents the network from memorizing the training set at the expense of generalization.

The **classification threshold** (default 0.5) converts a predicted probability into a class label. Lowering the threshold increases recall (catches more positives) at the cost of precision, and raising it does the opposite — this trade-off is explored interactively later in the notebook.

### Advantages and Applications

**Advantages**

- The perceptron is easy to understand, fast to train, and a useful linear baseline.
- MLPs are far more expressive and can approximate complex, non-linear decision boundaries.
- Early stopping controls overfitting without needing a separate regularization hyperparameter search.

**Disadvantages**

- A perceptron can only solve linearly separable problems.
- MLPs need more data, more tuning (layers, units, learning rate), and more computation.

**Applications**

These models are widely used as introductory neural networks for classification and as building blocks for larger deep learning systems in image recognition, natural language processing, and tabular data modeling.

```mermaid
flowchart LR
    X[Input Features] --> S[Standardization]
    S --> P[Perceptron: Single Dense Layer]
    S --> H1[Hidden Layer 1: ReLU]
    H1 --> H2[Hidden Layer 2: ReLU]
    H2 --> O[MLP Output Layer: Sigmoid]
    P --> M[Evaluation and Comparison]
    O --> M
```


## **Required Software and Libraries**

Install or prepare the following before starting the lab:

- Python 3.12 or later
- Jupyter Notebook, JupyterLab, VS Code, or Google Colab
- NumPy
- Pandas
- Matplotlib
- TensorFlow 2.x / Keras
- Scikit-learn
- Plotly and ipywidgets, used for the interactive visualizations in this notebook

### Starter Check

Run the first code cell (library imports and setup). If it prints `Setup complete. You can start the experiment.`, your environment is ready.

## **Dataset Description**

This experiment uses the **Breast Cancer Wisconsin dataset** again because it is a standard binary classification benchmark that works well with perceptron and MLP models, and its numeric features can be standardized cleanly before training.

- **Features:** 30 numeric measurements computed from breast mass images.
- **Target labels:** malignant and benign.
- **Source:** `sklearn.datasets.load_breast_cancer`.
- **Reason for selection:** a realistic binary task that highlights the difference between a linear decision boundary (perceptron) and a non-linear one (MLP).

## **Experimental Procedure**

Follow the steps below in order:

1. Load the dataset, split it into training, validation, and test sets, and standardize the features.
2. Build and train a single-neuron perceptron with early stopping.
3. Build and train an MLP with two hidden layers, also with early stopping.
4. Evaluate both models on the test set (accuracy, precision, recall, F1-score, ROC-AUC).
5. Compare interactive learning curves, confusion matrices, and ROC curves for both models.
6. Use the interactive threshold slider to explore the precision/recall trade-off for the MLP.

### Student Checkpoint

Before moving to the discussion section, make sure you can answer:

- Why is a separate validation set used during training instead of only a train/test split?
- At what epoch did early stopping halt training for each model, and what does that tell you about how quickly each model converged?
- What happens to the MLP's precision and recall as you drag the interactive threshold slider away from 0.5?


## **Source Code**

Run the following cells one by one. Read the output after each cell and add comments in your own words where instructed by your teacher.

**Tip:** Do not only run the notebook. Hover over the interactive Plotly charts to read exact values, and move the threshold slider at the end of the notebook to see how the MLP's confusion matrix changes.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # suppress verbose TensorFlow startup logs

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, FloatSlider

from sklearn.datasets import load_breast_cancer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.keras.utils.set_random_seed(42)
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

print("Setup complete. You can start the experiment.")

Setup complete. You can start the experiment.


In [2]:
# Load the Breast Cancer dataset

cancer = load_breast_cancer(as_frame=True)
cancer_df = cancer.frame
cancer_df

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,0
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,0
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,0
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,0


In [3]:
X = cancer.data
y = cancer.target

print("Shape:", X.shape)
print("Class counts:\n", y.value_counts().sort_index())
print("Target names:", cancer.target_names)

Shape: (569, 30)
Class counts:
 target
0    212
1    357
Name: count, dtype: int64
Target names: ['malignant' 'benign']


In [4]:
# Train / Validation / Test Split, then Standardize

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train_scaled.shape)
print("Validation shape:", X_val_scaled.shape)
print("Test shape:", X_test_scaled.shape)

Train shape: (364, 30)
Validation shape: (91, 30)
Test shape: (114, 30)


In [5]:
# Build the Perceptron Model

perceptron_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

perceptron_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

perceptron_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 1)              │            31 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31 (124.00 B)

 Trainable params: 31 (124.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
# Train the Perceptron with Early Stopping

perceptron_history = perceptron_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=40,
    batch_size=32,
    verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
)

print(f"Perceptron training finished after {len(perceptron_history.history['loss'])} epochs.")

Perceptron training finished after 40 epochs.


In [7]:
# Evaluate the Perceptron on the Test Set

perceptron_test_probs = perceptron_model.predict(X_test_scaled, verbose=0).ravel()
perceptron_test_pred = (perceptron_test_probs >= 0.5).astype(int)

perceptron_accuracy = accuracy_score(y_test, perceptron_test_pred)
print(f"Perceptron Test Accuracy : {perceptron_accuracy:.4f}")
print(classification_report(y_test, perceptron_test_pred, target_names=cancer.target_names))

Perceptron Test Accuracy : 0.9737
              precision    recall  f1-score   support

   malignant       0.95      0.98      0.96        42
      benign       0.99      0.97      0.98        72

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



In [8]:
# Build the MLP Model

mlp_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

mlp_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

mlp_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 32)             │           992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,537 (6.00 KB)

 Trainable params: 1,537 (6.00 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# Train the MLP with Early Stopping

mlp_history = mlp_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=60,
    batch_size=32,
    verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True)],
)

print(f"MLP training finished after {len(mlp_history.history['loss'])} epochs.")

MLP training finished after 60 epochs.


In [10]:
# Evaluate the MLP on the Test Set

mlp_test_probs = mlp_model.predict(X_test_scaled, verbose=0).ravel()
mlp_test_pred = (mlp_test_probs >= 0.5).astype(int)

mlp_accuracy = accuracy_score(y_test, mlp_test_pred)
print(f"MLP Test Accuracy : {mlp_accuracy:.4f}")
print(classification_report(y_test, mlp_test_pred, target_names=cancer.target_names))

MLP Test Accuracy : 0.9474
              precision    recall  f1-score   support

   malignant       0.91      0.95      0.93        42
      benign       0.97      0.94      0.96        72

    accuracy                           0.95       114
   macro avg       0.94      0.95      0.94       114
weighted avg       0.95      0.95      0.95       114



In [11]:
# Compare Both Models on Test Metrics

comparison = []
for name, y_pred, y_prob in [
    ("Perceptron", perceptron_test_pred, perceptron_test_probs),
    ("MLP", mlp_test_pred, mlp_test_probs),
]:
    comparison.append([
        name,
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_prob),
    ])

comparison_df = pd.DataFrame(
    comparison,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]
).round(4)

comparison_df

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Perceptron,0.9737,0.9859,0.9722,0.9790,0.9960
1,MLP,0.9474,0.9714,0.9444,0.9577,0.9927


In [12]:
# Interactive Learning Curves: Accuracy (hover for exact values)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Perceptron Accuracy", "MLP Accuracy"))

fig.add_trace(go.Scatter(y=perceptron_history.history["accuracy"], mode="lines", name="Train",
                          line=dict(color="#2a9d8f")), row=1, col=1)
fig.add_trace(go.Scatter(y=perceptron_history.history["val_accuracy"], mode="lines", name="Validation",
                          line=dict(color="#e76f51")), row=1, col=1)

fig.add_trace(go.Scatter(y=mlp_history.history["accuracy"], mode="lines", name="Train",
                          line=dict(color="#2a9d8f"), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(y=mlp_history.history["val_accuracy"], mode="lines", name="Validation",
                          line=dict(color="#e76f51"), showlegend=False), row=1, col=2)

fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="Accuracy")
fig.update_layout(title="Training vs Validation Accuracy", height=450, width=950,
                   template="plotly_white")

fig.show()

In [13]:
# Interactive Learning Curves: Loss (hover for exact values)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Perceptron Loss", "MLP Loss"))

fig.add_trace(go.Scatter(y=perceptron_history.history["loss"], mode="lines", name="Train",
                          line=dict(color="#2a9d8f")), row=1, col=1)
fig.add_trace(go.Scatter(y=perceptron_history.history["val_loss"], mode="lines", name="Validation",
                          line=dict(color="#e76f51")), row=1, col=1)

fig.add_trace(go.Scatter(y=mlp_history.history["loss"], mode="lines", name="Train",
                          line=dict(color="#2a9d8f"), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(y=mlp_history.history["val_loss"], mode="lines", name="Validation",
                          line=dict(color="#e76f51"), showlegend=False), row=1, col=2)

fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="Binary Cross-Entropy Loss")
fig.update_layout(title="Training vs Validation Loss", height=450, width=950,
                   template="plotly_white")

fig.show()

In [14]:
# Interactive Confusion Matrices (hover for exact counts)

perceptron_cm = confusion_matrix(y_test, perceptron_test_pred)
mlp_cm = confusion_matrix(y_test, mlp_test_pred)
labels = list(cancer.target_names)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Perceptron", "MLP"))

fig.add_trace(go.Heatmap(z=perceptron_cm, x=labels, y=labels, colorscale="Blues",
                          text=perceptron_cm, texttemplate="%{text}", showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=mlp_cm, x=labels, y=labels, colorscale="Blues",
                          text=mlp_cm, texttemplate="%{text}", showscale=False), row=1, col=2)

fig.update_xaxes(title_text="Predicted Label")
fig.update_yaxes(title_text="True Label", autorange="reversed")
fig.update_layout(title="Confusion Matrix Comparison", height=450, width=950,
                   template="plotly_white")

fig.show()

In [15]:
# Interactive ROC Curve Comparison (hover for threshold, TPR, FPR)

fpr_p, tpr_p, thr_p = roc_curve(y_test, perceptron_test_probs)
fpr_m, tpr_m, thr_m = roc_curve(y_test, mlp_test_probs)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=fpr_p, y=tpr_p, mode="lines", name=f"Perceptron (AUC = {roc_auc_score(y_test, perceptron_test_probs):.3f})",
    line=dict(color="#2a9d8f"),
    hovertemplate="FPR: %{x:.3f}<br>TPR: %{y:.3f}<extra></extra>"
))
fig.add_trace(go.Scatter(
    x=fpr_m, y=tpr_m, mode="lines", name=f"MLP (AUC = {roc_auc_score(y_test, mlp_test_probs):.3f})",
    line=dict(color="#e76f51"),
    hovertemplate="FPR: %{x:.3f}<br>TPR: %{y:.3f}<extra></extra>"
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines", name="Random Guess",
    line=dict(color="gray", dash="dash")
))

fig.update_layout(
    title="ROC Curve Comparison",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    height=500, width=650,
    template="plotly_white"
)

fig.show()

In [16]:
# Interactive Threshold Explorer for the MLP
#
# Move the slider to change the classification threshold applied to the MLP's
# predicted probabilities, and watch accuracy, precision, recall, F1-score,
# and the confusion matrix update immediately.

def explore_threshold(threshold=0.50):
    y_pred_t = (mlp_test_probs >= threshold).astype(int)

    acc = accuracy_score(y_test, y_pred_t)
    prec = precision_score(y_test, y_pred_t, zero_division=0)
    rec = recall_score(y_test, y_pred_t, zero_division=0)
    f1 = f1_score(y_test, y_pred_t, zero_division=0)
    cm = confusion_matrix(y_test, y_pred_t)

    print(f"Threshold : {threshold:.2f}")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-score  : {f1:.4f}")

    fig = go.Figure(data=go.Heatmap(
        z=cm, x=labels, y=labels, colorscale="Blues",
        text=cm, texttemplate="%{text}", showscale=False
    ))
    fig.update_xaxes(title_text="Predicted Label")
    fig.update_yaxes(title_text="True Label", autorange="reversed")
    fig.update_layout(
        title=f"MLP Confusion Matrix at Threshold = {threshold:.2f}",
        height=420, width=420, template="plotly_white"
    )
    fig.show()

interact(
    explore_threshold,
    threshold=FloatSlider(value=0.50, min=0.05, max=0.95, step=0.05,
                           description="Threshold", continuous_update=False)
);

interactive(children=(FloatSlider(value=0.5, continuous_update=False, description='Threshold', max=0.95, min=0…

## **Expected Output**

After running all cells, the notebook should display:

- Dataset shape, class counts, and target names.
- Train/validation/test split shapes after standardization.
- Model summaries and training-completion messages showing the epoch at which early stopping triggered.
- Test-set accuracy and a full classification report for both the perceptron and the MLP.
- A metric comparison table (accuracy, precision, recall, F1-score, ROC-AUC) for both models.
- Interactive (hoverable) accuracy curves, loss curves, confusion matrices, and an ROC curve comparison.
- An interactive threshold slider that updates the MLP's accuracy, precision, recall, F1-score, and confusion matrix live.

You should include the important outputs — and a brief note on what you observed while moving the threshold slider — in your lab report.

## **Performance Evaluation**

Use accuracy, precision, recall, F1-score, ROC-AUC, and confusion matrices to evaluate both models. Learning curves are especially important because they indicate whether a model is overfitting (validation loss rising while training loss falls) or underfitting (both losses staying high). Use the interactive threshold explorer to evaluate the precision/recall trade-off beyond the default 0.5 cutoff.

## **Discussion**

The perceptron is a simple linear classifier, so it struggles when the classes are not perfectly linearly separable, though on this dataset it still performs reasonably well because the classes are close to linearly separable after standardization. The MLP adds non-linearity through hidden layers and therefore captures more complex patterns, usually achieving equal or better accuracy and a smoother learning curve. Early stopping helps prevent unnecessary training and gives a better estimate of generalization by restoring the weights from the epoch with the best validation loss. The interactive threshold explorer shows directly that moving the cutoff away from 0.5 trades precision for recall, which matters in a medical context where missed malignant cases (false negatives) are usually costlier than false alarms.

## **Viva Questions**

1. **Q:** What is the perceptron model?  
   **A:** A single-layer neural classifier that computes a weighted sum of inputs and outputs a binary decision.

2. **Q:** What makes an MLP more powerful than a perceptron?  
   **A:** Hidden layers with non-linear activations allow it to model more complex, non-linearly-separable relationships.

3. **Q:** What is backpropagation?  
   **A:** A method for computing gradients of the loss with respect to every weight in the network, layer by layer, so weights can be updated efficiently.

4. **Q:** Why do we standardize input features before training?  
   **A:** To put all features on a comparable scale, which improves training stability and convergence speed.

5. **Q:** What is early stopping, and what does `restore_best_weights` do?  
   **A:** Early stopping halts training once validation performance stops improving for a set number of epochs (patience); `restore_best_weights` reloads the weights from the best-performing epoch rather than the last one.

6. **Q:** Why is the sigmoid activation used in the output layer for binary classification?  
   **A:** It maps the raw output to a probability-like value between 0 and 1.

7. **Q:** What is the purpose of a validation set, separate from the test set?  
   **A:** To tune the model and monitor generalization during training, without letting information from the final test set leak into training decisions.

8. **Q:** Why can a perceptron fail on some datasets?  
   **A:** Because it can only learn a linear decision boundary and cannot separate classes that require a non-linear boundary.

9. **Q:** What happens to precision and recall as the classification threshold is lowered?  
   **A:** Recall tends to increase (more positives are captured) while precision tends to decrease (more false positives are included).

10. **Q:** Why are neural networks useful in AI?  
    **A:** They can learn complex, non-linear mappings directly from data and serve as the foundation for deeper architectures used across AI.

## **Exercises**

Complete the following tasks after running the notebook:

1. Increase the number of hidden layers or units in the MLP and compare the results.
2. Try a different learning rate for the MLP's optimizer and observe the effect on convergence speed.
3. Replace the `relu` activation with `tanh` in the hidden layers and evaluate the change in accuracy.
4. Use the interactive threshold slider to find the threshold that maximizes F1-score, and report it.
5. Add a `Dropout` layer to the MLP and check whether validation accuracy or the gap between train/validation loss changes.
6. Use the same workflow on the Iris dataset (with a softmax output layer) for a multiclass experiment.

## **Lab Report Submission Checklist**

Before submitting, confirm that your report includes:

- Completed cover information.
- Objective and theory summary in your own words.
- All important code outputs, including the metric comparison table.
- Screenshots or descriptions of the interactive learning curves, confusion matrices, ROC curve, and threshold explorer.
- Short discussion of which model performed best and why, and what you observed while moving the threshold slider.
- Answers to assigned viva questions.
- Completed exercises.

**Reminder:** A good lab report should not only show screenshots. It should explain what the outputs mean.
